# Feature Selection

The notebook fits masks on a training partition and evaluates them without using held-out labels.

In [ ]:
from pathlib import Path
import sys

LESSON_REL = Path('phases/02-ml-fundamentals/18-feature-selection')
MODULE_FILE = 'feature_selection.py'
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [Path.cwd() / LESSON_REL / 'code', Path.cwd() / 'code']
candidates.extend(root / LESSON_REL / 'code' for root in roots)
candidates.extend(root / 'code' for root in roots)
CODE = next((candidate.resolve() for candidate in candidates if (candidate / MODULE_FILE).is_file()), None)
if CODE is None:
    raise RuntimeError('Could not locate ' + str(LESSON_REL / 'code' / MODULE_FILE))
sys.path.insert(0, str(CODE))

## Build It

Run the next cell from the lesson directory; all values are local, deterministic fixtures.

In [ ]:
import numpy as np
import feature_selection as fs

X, y, names = fs.make_feature_selection_data(80, seed=7)
split = 64
var_mask, variances = fs.variance_threshold(X[:split], threshold=0.01)
mi = fs.mutual_information(X[:split], y[:split], n_bins=6)
rfe_mask, ranks = fs.rfe(X[:split], y[:split], n_features_to_select=5, epochs=30)
tree = fs.tree_importance(X[:split], y[:split], n_trees=8, max_depth=3, seed=7)
assert X.shape == (80, 20) and len(names) == 20
assert var_mask.shape == (20,) and np.isfinite(mi).all()
assert int(rfe_mask.sum()) == 5 and tree.shape == (20,)
try:
    fs.rfe(X[:split], np.array([0, 2] * (split // 2)), n_features_to_select=5, epochs=2)
except ValueError:
    pass
else:
    raise AssertionError('feature selectors require numeric 0/1 labels')

## Exercises and Ship It

Compare the selected mask with an all-feature baseline on the held-out rows and record how correlated columns exchange importance across seeds.

Record the observed shape/value and update the lesson output card. A notebook result is evidence for this fixture, not a production guarantee.